# D396 — Snowflake view family

A view can be a saved query, a protected interface, a stored acceleration structure, a refreshed transformation, or a business model. Those objects look similar when queried, but they behave differently when source data changes.

This notebook uses one small sales dataset to teach **standard views, secure views, materialized views, dynamic tables, and semantic views**. Every fenced block is designed for direct copy/paste into a Snowsight SQL worksheet.

## 1. Learning objectives

After completing the notebook, you should be able to:

1. explain whether each object stores a result and when its query runs;
2. choose among a view, materialized view, and dynamic table;
3. expose data safely through a secure view without assuming that `SECURE` masks columns;
4. change source data and predict what each object returns;
5. model business dimensions, facts, metrics, and relationships in a semantic view;
6. inspect, alter, grant, and remove each object.

## 2. The complete mental model

| Object | Stores the query result? | Evaluation or refresh | Source-query shape | How it is used | Direct DML on object? | Main tradeoff |
|---|---:|---|---|---|---:|---|
| Standard view | No | Its `SELECT` runs when referenced | Flexible, including joins and nested logic | Query the view name | No | Always reads current sources, but repeats work |
| Secure view | No | Query time | Same broad model as a standard view, with optimizer protections | Query the view name | No | Protects sensitive logic/data from indirect exposure; can be slower |
| Materialized view | Yes | Snowflake maintains it automatically | Restricted; one base table and no joins | Query it directly, or let the optimizer rewrite a base-table query | No | Faster repeated work, with storage and maintenance cost |
| Dynamic table | Yes | Automated refresh toward `TARGET_LAG` using a named warehouse | Supports useful multi-table transformations and pipelines | Query the dynamic-table name explicitly | No | Declarative pipeline freshness, with refresh compute and some lag |
| Semantic view | Stores the logical model, not a general cached query result | A semantic query is compiled against physical tables | Declared tables, relationships, facts, dimensions, and metrics | `SEMANTIC_VIEW(...)` query construct | No | Consistent business meaning; requires model design and feature availability |

**The critical distinction:** a standard view saves SQL, a materialized view saves and maintains a restricted query result for acceleration, and a dynamic table saves and refreshes a transformation result to a freshness objective. Snowflake can transparently rewrite a compatible base-table query to use a materialized view. It does not transparently substitute a dynamic table; consumers query the dynamic table explicitly.

`SECURE` is a protection property, not a separate storage engine. Both standard and materialized views can be secure.

Sources: [comparison of views, materialized views, and dynamic tables](https://docs.snowflake.com/en/user-guide/overview-view-mview-dts), [secure views](https://docs.snowflake.com/en/user-guide/views-secure), [semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/overview).

## 3. Build the shared lab data

Run this setup once. It creates a small warehouse, two source tables, and repeatable inline data. `TRUNCATE` plus `INSERT` keeps the table identities stable if derived objects from an earlier run still exist.

```sql
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS D39_TABLE_LAB;
CREATE SCHEMA IF NOT EXISTS D39_TABLE_LAB.VIEWS;
CREATE WAREHOUSE IF NOT EXISTS D39_LAB_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE D39_LAB_WH;
USE DATABASE D39_TABLE_LAB;
USE SCHEMA VIEWS;

CREATE TABLE IF NOT EXISTS DIM_CUSTOMER (
  CUSTOMER_ID INTEGER,
  CUSTOMER_NAME VARCHAR(100),
  EMAIL VARCHAR(255),
  SEGMENT VARCHAR(30),
  REGION VARCHAR(30),
  IS_ACTIVE BOOLEAN
);

CREATE TABLE IF NOT EXISTS FACT_SALES (
  SALE_ID INTEGER,
  CUSTOMER_ID INTEGER,
  SALE_DATE DATE,
  PRODUCT VARCHAR(100),
  QUANTITY INTEGER,
  AMOUNT NUMBER(12,2),
  STATUS VARCHAR(20)
);

TRUNCATE TABLE DIM_CUSTOMER;
TRUNCATE TABLE FACT_SALES;

INSERT INTO DIM_CUSTOMER VALUES
  (101, 'Asha Rao',  'asha@example.com',  'ENTERPRISE', 'SOUTH', TRUE),
  (102, 'Ben Lee',   'ben@example.com',   'SMB',        'WEST',  TRUE),
  (103, 'Carla Diaz','carla@example.com', 'ENTERPRISE', 'EAST',  TRUE),
  (104, 'Dev Shah',  'dev@example.com',   'CONSUMER',   'NORTH', FALSE);

INSERT INTO FACT_SALES VALUES
  (1001, 101, '2026-09-01', 'Laptop',   1, 1200.00, 'COMPLETE'),
  (1002, 101, '2026-09-01', 'Dock',     2,  380.00, 'COMPLETE'),
  (1003, 102, '2026-09-02', 'Monitor',  1,  450.00, 'COMPLETE'),
  (1004, 103, '2026-09-02', 'Keyboard', 4,  400.00, 'PENDING'),
  (1005, 103, '2026-09-03', 'Laptop',   2, 2500.00, 'COMPLETE'),
  (1006, 104, '2026-09-03', 'Mouse',    1,   45.00, 'CANCELLED'),
  (1007, 102, '2026-09-04', 'Headset',  3,  270.00, 'COMPLETE'),
  (1008, 103, '2026-09-04', 'Dock',     1,  190.00, 'COMPLETE');

SELECT * FROM DIM_CUSTOMER ORDER BY CUSTOMER_ID;
SELECT * FROM FACT_SALES ORDER BY SALE_ID;
```

## 4. Standard view — a saved, live query

A standard view stores its definition. It does not store the rows returned by that definition. Every reference evaluates the view against its current source data. A view is read-only, so perform `INSERT`, `UPDATE`, `DELETE`, `MERGE`, and `TRUNCATE` against its source tables.

Use a standard view to give a query a stable name, simplify joins, expose selected rows or columns, or create a logical interface over changing implementation details.

```sql
CREATE OR REPLACE VIEW V_CUSTOMER_SALES
  COMMENT = 'Completed sales with customer attributes'
AS
SELECT
  S.SALE_ID,
  S.SALE_DATE,
  C.CUSTOMER_ID,
  C.CUSTOMER_NAME,
  C.SEGMENT,
  C.REGION,
  S.PRODUCT,
  S.QUANTITY,
  S.AMOUNT
FROM FACT_SALES AS S
JOIN DIM_CUSTOMER AS C
  ON S.CUSTOMER_ID = C.CUSTOMER_ID
WHERE S.STATUS = 'COMPLETE';

SELECT *
FROM V_CUSTOMER_SALES
ORDER BY SALE_ID;

SELECT REGION, SUM(AMOUNT) AS REVENUE
FROM V_CUSTOMER_SALES
GROUP BY REGION
ORDER BY REVENUE DESC;
```

Prefer explicit column lists in durable views. `SELECT *` is captured when the view is created; later source-column changes do not safely redesign the view for you. Snowflake limits a view definition to 1 MB and view nesting to 20 levels. Source: [`CREATE VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-view).

### Observe source changes through the view

The next block demonstrates insert, update, and delete. The view reflects every committed change the next time it runs. The final update also shows how a view can identify rows while the DML targets the base table.

```sql
-- INSERT into the source.
INSERT INTO FACT_SALES VALUES
  (1009, 102, '2026-09-05', 'Webcam', 1, 125.00, 'COMPLETE');

-- UPDATE the source. This sale disappears because the view filters COMPLETE.
UPDATE FACT_SALES
SET STATUS = 'CANCELLED'
WHERE SALE_ID = 1003;

-- DELETE from the source.
DELETE FROM FACT_SALES
WHERE SALE_ID = 1007;

SELECT * FROM V_CUSTOMER_SALES ORDER BY SALE_ID;

-- The view may be used as a selector while the base table receives the DML.
UPDATE DIM_CUSTOMER
SET SEGMENT = 'MID_MARKET'
WHERE CUSTOMER_ID IN (
  SELECT CUSTOMER_ID
  FROM V_CUSTOMER_SALES
  WHERE REGION = 'WEST'
);

SELECT * FROM V_CUSTOMER_SALES ORDER BY SALE_ID;

-- Unsupported: the view itself is read-only.
-- UPDATE V_CUSTOMER_SALES SET AMOUNT = 0 WHERE SALE_ID = 1001;
```

Source: [overview of views](https://docs.snowflake.com/en/user-guide/views-introduction).

### Replace, alter, inspect, grant, and create variants

`CREATE OR REPLACE VIEW` changes the query definition. Add `COPY GRANTS` when a replacement should retain existing privileges. `ALTER VIEW` manages properties such as comments and security mode; it does not rewrite the `SELECT`. A temporary view has session scope. A recursive view can reference itself and must have an explicit output column list.

```sql
-- Add a derived column by replacing the definition.
CREATE OR REPLACE VIEW V_CUSTOMER_SALES
  COPY GRANTS
AS
SELECT
  S.SALE_ID, S.SALE_DATE, C.CUSTOMER_ID, C.CUSTOMER_NAME,
  C.SEGMENT, C.REGION, S.PRODUCT, S.QUANTITY, S.AMOUNT,
  IFF(S.AMOUNT >= 1000, 'HIGH_VALUE', 'STANDARD') AS VALUE_BAND
FROM FACT_SALES AS S
JOIN DIM_CUSTOMER AS C ON S.CUSTOMER_ID = C.CUSTOMER_ID
WHERE S.STATUS = 'COMPLETE';

ALTER VIEW V_CUSTOMER_SALES
  SET COMMENT = 'Completed customer sales with value band';

DESCRIBE VIEW V_CUSTOMER_SALES;
SHOW VIEWS LIKE 'V_CUSTOMER_SALES' IN SCHEMA D39_TABLE_LAB.VIEWS;
SELECT GET_DDL('VIEW', 'D39_TABLE_LAB.VIEWS.V_CUSTOMER_SALES');

-- Replace ANALYST_ROLE with an existing role before running this grant.
-- GRANT SELECT ON VIEW V_CUSTOMER_SALES TO ROLE ANALYST_ROLE;

CREATE OR REPLACE TEMPORARY VIEW V_MY_SESSION_SALES AS
SELECT * FROM FACT_SALES WHERE STATUS = 'COMPLETE';

CREATE OR REPLACE RECURSIVE VIEW V_SEVEN_DAYS (DAY_NO, CALENDAR_DATE) AS
  SELECT 1, CURRENT_DATE()
  UNION ALL
  SELECT DAY_NO + 1, DATEADD('DAY', 1, CALENDAR_DATE)
  FROM V_SEVEN_DAYS
  WHERE DAY_NO < 7;

SELECT * FROM V_SEVEN_DAYS ORDER BY DAY_NO;
```

## 5. Secure view — a protected query boundary

A secure view still computes its result at query time. Snowflake hides its definition and some internal details from unauthorized users, and it disables optimizer techniques that could expose values indirectly. Those protections can reduce performance. Use `SECURE` when the privacy boundary requires it.

A secure view does **not** automatically mask, hash, filter, or tokenize sensitive values. Its SQL must explicitly choose the safe output, and privileges must still be granted correctly. Only the owner and appropriately authorized roles can see the definition.

```sql
CREATE OR REPLACE SECURE VIEW V_CUSTOMER_CONTACT_SECURE
  COMMENT = 'Active customers with masked contact data'
AS
SELECT
  CUSTOMER_ID,
  CUSTOMER_NAME,
  REGEXP_REPLACE(EMAIL, '^[^@]+', '***') AS MASKED_EMAIL,
  SEGMENT,
  REGION
FROM DIM_CUSTOMER
WHERE IS_ACTIVE;

SELECT * FROM V_CUSTOMER_CONTACT_SECURE ORDER BY CUSTOMER_ID;

SHOW VIEWS LIKE 'V_CUSTOMER_CONTACT_SECURE'
  IN SCHEMA D39_TABLE_LAB.VIEWS;

SELECT TABLE_NAME, IS_SECURE, COMMENT
FROM D39_TABLE_LAB.INFORMATION_SCHEMA.VIEWS
WHERE TABLE_SCHEMA = 'VIEWS'
  AND TABLE_NAME = 'V_CUSTOMER_CONTACT_SECURE';

-- The owner can change security mode without changing the SELECT.
ALTER VIEW V_CUSTOMER_CONTACT_SECURE UNSET SECURE;
ALTER VIEW V_CUSTOMER_CONTACT_SECURE SET SECURE;

-- Replace DATA_CONSUMER_ROLE with an existing role. The consumer needs
-- database/schema access and SELECT on the view, not SELECT on base tables.
-- GRANT USAGE ON DATABASE D39_TABLE_LAB TO ROLE DATA_CONSUMER_ROLE;
-- GRANT USAGE ON SCHEMA D39_TABLE_LAB.VIEWS TO ROLE DATA_CONSUMER_ROLE;
-- GRANT SELECT ON VIEW V_CUSTOMER_CONTACT_SECURE TO ROLE DATA_CONSUMER_ROLE;
```

Standard and materialized views can both be secure. For example, an Enterprise account can use `CREATE SECURE MATERIALIZED VIEW ...`. Source: [working with secure views](https://docs.snowflake.com/en/user-guide/views-secure).

## 6. Materialized view — maintained acceleration

A materialized view stores a precomputed result and Snowflake maintains it when its single base table changes. It is an **Enterprise Edition** feature. The query service returns current results: when maintenance has not yet incorporated a base-table change, Snowflake can combine the materialized data with outstanding base-table changes.

Choose it when the same expensive filter or aggregation runs frequently, the result is much smaller than the base table, and the base table changes less often than it is queried. Storage and maintenance consume credits. Test the benefit before keeping one.

```sql
CREATE OR REPLACE MATERIALIZED VIEW MV_DAILY_SALES AS
SELECT
  SALE_DATE,
  CUSTOMER_ID,
  COUNT(*) AS ORDER_COUNT,
  SUM(QUANTITY) AS UNITS_SOLD,
  SUM(AMOUNT) AS REVENUE
FROM FACT_SALES
WHERE STATUS = 'COMPLETE'
GROUP BY SALE_DATE, CUSTOMER_ID;

SELECT * FROM MV_DAILY_SALES ORDER BY SALE_DATE, CUSTOMER_ID;
SHOW MATERIALIZED VIEWS IN SCHEMA D39_TABLE_LAB.VIEWS;
```

If creation fails because the account edition does not include materialized views, study the SQL and continue at the dynamic-table section. Source: [materialized views](https://docs.snowflake.com/en/user-guide/views-materialized).

### Change data, inspect rewrite, and manage maintenance

DML targets `FACT_SALES`; the materialized view itself is read-only. Snowflake may transparently use `MV_DAILY_SALES` for a compatible query written against `FACT_SALES`. The optimizer can still choose the base table for this tiny lab, so inspect the plan rather than assuming a rewrite occurred.

```sql
INSERT INTO FACT_SALES VALUES
  (1010, 101, '2026-09-05', 'Tablet', 2, 1400.00, 'COMPLETE');

UPDATE FACT_SALES
SET AMOUNT = 1300.00
WHERE SALE_ID = 1001;

DELETE FROM FACT_SALES
WHERE SALE_ID = 1002;

SELECT * FROM MV_DAILY_SALES ORDER BY SALE_DATE, CUSTOMER_ID;

-- Ask for the same shape through the base table. Look for MV usage in
-- Query Profile or the plan; small data might make a base scan cheaper.
EXPLAIN USING TEXT
SELECT SALE_DATE, CUSTOMER_ID, COUNT(*) AS ORDER_COUNT,
       SUM(QUANTITY) AS UNITS_SOLD, SUM(AMOUNT) AS REVENUE
FROM FACT_SALES
WHERE STATUS = 'COMPLETE'
GROUP BY SALE_DATE, CUSTOMER_ID;

ALTER MATERIALIZED VIEW MV_DAILY_SALES SUSPEND;
ALTER MATERIALIZED VIEW MV_DAILY_SALES RESUME;
ALTER MATERIALIZED VIEW MV_DAILY_SALES
  SET COMMENT = 'Daily completed-sales acceleration';

-- Unsupported: change FACT_SALES instead.
-- UPDATE MV_DAILY_SALES SET REVENUE = 0;
```

### Materialized-view boundaries

The defining query must use one base table. It cannot contain joins, self-joins, window functions, `HAVING`, `ORDER BY`, `LIMIT`, nested subqueries, set operators, or unsupported/non-deterministic functions. It also cannot read a standard view, another materialized view, a hybrid table, or a dynamic table.

```sql
-- Invalid materialized-view design: it joins two tables.
-- Use a dynamic table for this stored multi-table transformation.
-- CREATE MATERIALIZED VIEW MV_INVALID_CUSTOMER_SALES AS
-- SELECT C.REGION, SUM(S.AMOUNT) AS REVENUE
-- FROM FACT_SALES AS S
-- JOIN DIM_CUSTOMER AS C ON S.CUSTOMER_ID = C.CUSTOMER_ID
-- GROUP BY C.REGION;
```

Full rules: [`CREATE MATERIALIZED VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-materialized-view).

## 7. Dynamic table — a refreshed transformation

A dynamic table materializes its query result. Snowflake refreshes it toward a target lag using the named warehouse. `TARGET_LAG = '5 minutes'` expresses a freshness objective relative to its source data; it is not a cron schedule and not a guarantee that every refresh starts exactly five minutes apart.

Unlike the materialized view, this example can store a join between the fact and dimension tables. Query the dynamic table explicitly; Snowflake does not transparently rewrite an arbitrary source query to use it.

```sql
CREATE OR REPLACE DYNAMIC TABLE DT_REGION_DAILY_SALES
  TARGET_LAG = '5 minutes'
  WAREHOUSE = D39_LAB_WH
  REFRESH_MODE = AUTO
  INITIALIZE = ON_CREATE
AS
SELECT
  S.SALE_DATE,
  C.REGION,
  C.SEGMENT,
  COUNT(*) AS ORDER_COUNT,
  SUM(S.QUANTITY) AS UNITS_SOLD,
  SUM(S.AMOUNT) AS REVENUE
FROM FACT_SALES AS S
JOIN DIM_CUSTOMER AS C
  ON S.CUSTOMER_ID = C.CUSTOMER_ID
WHERE S.STATUS = 'COMPLETE'
GROUP BY S.SALE_DATE, C.REGION, C.SEGMENT;

SELECT * FROM DT_REGION_DAILY_SALES
ORDER BY SALE_DATE, REGION, SEGMENT;

SHOW DYNAMIC TABLES LIKE 'DT_REGION_DAILY_SALES'
  IN SCHEMA D39_TABLE_LAB.VIEWS;
```

Sources: [dynamic table overview](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview), [dynamic table limitations](https://docs.snowflake.com/en/user-guide/dynamic-tables-limitations).

### Change sources, refresh, suspend, and resume

A dynamic table exposes the last successfully published refresh. For a deterministic lab result, request a manual refresh after changing the sources. In production, automatic refreshes aim to keep the data within target lag.

```sql
INSERT INTO FACT_SALES VALUES
  (1011, 103, '2026-09-05', 'Camera', 1, 800.00, 'COMPLETE');

UPDATE DIM_CUSTOMER
SET REGION = 'CENTRAL'
WHERE CUSTOMER_ID = 103;

DELETE FROM FACT_SALES
WHERE SALE_ID = 1005;

ALTER DYNAMIC TABLE DT_REGION_DAILY_SALES REFRESH;

SELECT * FROM DT_REGION_DAILY_SALES
ORDER BY SALE_DATE, REGION, SEGMENT;

ALTER DYNAMIC TABLE DT_REGION_DAILY_SALES SUSPEND;
ALTER DYNAMIC TABLE DT_REGION_DAILY_SALES RESUME;
ALTER DYNAMIC TABLE DT_REGION_DAILY_SALES
  SET TARGET_LAG = '10 minutes';

-- Unsupported: change FACT_SALES or DIM_CUSTOMER instead.
-- DELETE FROM DT_REGION_DAILY_SALES WHERE REGION = 'WEST';
```

## 8. View → materialized view → dynamic table

Use these questions in order:

| Question | If yes | Why |
|---|---|---|
| Do you only need a reusable logical query with current source data? | Standard view | No stored result or refresh system is needed |
| Must the interface protect sensitive logic or reduce inference risk? | Add `SECURE` to a standard view | The result still runs at query time |
| Is a repeated, expensive query over one table compatible with materialized-view restrictions? | Materialized view | Snowflake maintains acceleration and may rewrite compatible base queries |
| Do you need stored joins, a transformation pipeline, or an explicit freshness objective? | Dynamic table | It materializes broader transformations and refreshes toward target lag |

A materialized view is not simply a faster standard view, and a dynamic table is not simply a materialized view with a schedule. Their query restrictions, optimizer integration, freshness behavior, and intended workloads differ.

### What source changes mean

1. **View:** the next query executes the definition and sees committed source changes.
2. **Materialized view:** Snowflake maintains the stored result and preserves current query semantics.
3. **Dynamic table:** a refresh publishes a new snapshot; consumers can see the previous snapshot until that refresh completes.

## 9. Semantic view — a governed business model

A semantic view defines the language of the business over physical tables:

- **logical tables** identify the physical sources and keys;
- **relationships** declare how logical tables connect;
- **facts** are row-level numeric or measurable expressions;
- **dimensions** describe how results can be grouped or filtered;
- **metrics** define governed aggregations such as revenue or order count.

The semantic definition is a logical model, not a general cached result like a materialized view or dynamic table. Newer Snowflake releases also support optional semantic-view materializations for acceleration; that separate optimization does not change the core model. Semantic-view availability can depend on account rollout and privileges.

Source: [semantic view overview](https://docs.snowflake.com/en/user-guide/views-semantic/overview).

### Create a semantic view

Clause order matters: `TABLES`, `RELATIONSHIPS`, `FACTS`, `DIMENSIONS`, then `METRICS`. References use the logical aliases declared in `TABLES`. At least one dimension or metric is required.

```sql
CREATE OR REPLACE SEMANTIC VIEW SALES_SEMANTIC
  TABLES (
    CUSTOMERS AS D39_TABLE_LAB.VIEWS.DIM_CUSTOMER
      PRIMARY KEY (CUSTOMER_ID)
      WITH SYNONYMS ('BUYERS')
      COMMENT = 'Customer master data',
    SALES AS D39_TABLE_LAB.VIEWS.FACT_SALES
      PRIMARY KEY (SALE_ID)
      COMMENT = 'Individual sales transactions'
  )
  RELATIONSHIPS (
    SALES_CUSTOMER AS SALES(CUSTOMER_ID) REFERENCES CUSTOMERS
  )
  FACTS (
    SALES.SALE_AMOUNT AS SALES.AMOUNT
      COMMENT = 'Recorded transaction amount',
    SALES.UNITS AS SALES.QUANTITY
      COMMENT = 'Units on the transaction'
  )
  DIMENSIONS (
    CUSTOMERS.CUSTOMER_NAME AS CUSTOMERS.CUSTOMER_NAME
      WITH SYNONYMS ('BUYER NAME'),
    CUSTOMERS.SEGMENT AS CUSTOMERS.SEGMENT,
    CUSTOMERS.REGION AS CUSTOMERS.REGION,
    SALES.SALE_DATE AS SALES.SALE_DATE,
    SALES.PRODUCT AS SALES.PRODUCT,
    SALES.STATUS AS SALES.STATUS
  )
  METRICS (
    SALES.REVENUE AS SUM(SALES.SALE_AMOUNT)
      COMMENT = 'Sum of transaction amount',
    SALES.ORDER_COUNT AS COUNT(DISTINCT SALES.SALE_ID)
      COMMENT = 'Distinct sale count',
    SALES.UNITS_SOLD AS SUM(SALES.UNITS),
    SALES.AVERAGE_ORDER_VALUE AS AVG(SALES.SALE_AMOUNT)
  )
  COMMENT = 'Governed sales metrics by customer, date, product, and region';
```

Syntax reference: [`CREATE SEMANTIC VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view).

### Query the business model

Use `SEMANTIC_VIEW(...)`, name the dimensions and metrics you need, and filter using semantic fields. The model resolves the customer relationship and applies the governed metric definitions. A consumer granted `SELECT` on the semantic view does not also need `SELECT` on its underlying tables.

```sql
SELECT *
FROM SEMANTIC_VIEW(
  SALES_SEMANTIC
  DIMENSIONS SALES.SALE_DATE, CUSTOMERS.REGION
  METRICS SALES.REVENUE, SALES.ORDER_COUNT, SALES.UNITS_SOLD
  WHERE SALES.STATUS = 'COMPLETE'
)
ORDER BY SALE_DATE, REGION;

SELECT *
FROM SEMANTIC_VIEW(
  SALES_SEMANTIC
  DIMENSIONS CUSTOMERS.SEGMENT, SALES.PRODUCT
  METRICS SALES.REVENUE, SALES.AVERAGE_ORDER_VALUE
  WHERE SALES.STATUS = 'COMPLETE'
)
ORDER BY REVENUE DESC;
```

Query reference: [querying semantic views with SQL](https://docs.snowflake.com/en/user-guide/views-semantic/querying).

### Inspect and grant the semantic model

```sql
SHOW SEMANTIC VIEWS LIKE 'SALES_SEMANTIC'
  IN SCHEMA D39_TABLE_LAB.VIEWS;
DESCRIBE SEMANTIC VIEW SALES_SEMANTIC;
SHOW SEMANTIC FACTS IN SALES_SEMANTIC;
SHOW SEMANTIC DIMENSIONS IN SALES_SEMANTIC;
SHOW SEMANTIC METRICS IN SALES_SEMANTIC;
SHOW SEMANTIC DIMENSIONS IN SALES_SEMANTIC FOR METRIC REVENUE;

-- Replace BI_CONSUMER_ROLE with an existing role before running.
-- GRANT USAGE ON DATABASE D39_TABLE_LAB TO ROLE BI_CONSUMER_ROLE;
-- GRANT USAGE ON SCHEMA D39_TABLE_LAB.VIEWS TO ROLE BI_CONSUMER_ROLE;
-- GRANT SELECT ON SEMANTIC VIEW SALES_SEMANTIC TO ROLE BI_CONSUMER_ROLE;
```

## 10. Practice tasks

1. Create `V_OPEN_SALES` for `PENDING` transactions. Insert one pending sale and prove the view sees it without a refresh.
2. Build a secure view that exposes only customer ID, segment, and region. Explain why omitting email is stronger than merely hiding the view definition.
3. Propose a materialized view for completed sales by product. Check every expression against the single-table restrictions.
4. Create a second dynamic table from `DT_REGION_DAILY_SALES` with `TARGET_LAG = DOWNSTREAM` and aggregate to region. Observe how target lag works through a pipeline.
5. Add an `ACTIVE_CUSTOMER_COUNT` metric to the semantic model. Decide which logical table owns it and whether it needs another fact.
6. For each requirement below, choose an object and justify cost, freshness, and security:
   - always-current reusable join;
   - repeated expensive aggregation over one large fact table;
   - refreshed multi-table executive summary;
   - governed definition of revenue for multiple BI tools.

## 11. Optional cleanup

Drop objects from the most specialized dependents back to the sources. The guarded blocks let cleanup run even when an account does not support a feature used above. Keep the shared database if you plan to run other D39 notebooks.

```sql
USE ROLE SYSADMIN;
USE DATABASE D39_TABLE_LAB;
USE SCHEMA VIEWS;

BEGIN
  DROP SEMANTIC VIEW IF EXISTS SALES_SEMANTIC;
EXCEPTION
  WHEN OTHER THEN NULL;
END;

BEGIN
  DROP DYNAMIC TABLE IF EXISTS DT_REGION_DAILY_SALES;
EXCEPTION
  WHEN OTHER THEN NULL;
END;

BEGIN
  DROP MATERIALIZED VIEW IF EXISTS MV_DAILY_SALES;
EXCEPTION
  WHEN OTHER THEN NULL;
END;

DROP VIEW IF EXISTS V_CUSTOMER_CONTACT_SECURE;
DROP VIEW IF EXISTS V_CUSTOMER_SALES;
DROP VIEW IF EXISTS V_SEVEN_DAYS;
DROP TABLE IF EXISTS FACT_SALES;
DROP TABLE IF EXISTS DIM_CUSTOMER;

-- Remove everything created by the D39 module only when you are finished.
-- DROP DATABASE D39_TABLE_LAB;
-- DROP WAREHOUSE D39_LAB_WH;
```

## 12. Official references

- [Comparison: views, materialized views, and dynamic tables](https://docs.snowflake.com/en/user-guide/overview-view-mview-dts)
- [Overview of views](https://docs.snowflake.com/en/user-guide/views-introduction) and [`CREATE VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-view)
- [Secure views](https://docs.snowflake.com/en/user-guide/views-secure)
- [Materialized views](https://docs.snowflake.com/en/user-guide/views-materialized) and [`CREATE MATERIALIZED VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-materialized-view)
- [Dynamic tables](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview)
- [Semantic view example](https://docs.snowflake.com/en/user-guide/views-semantic/example), [`CREATE SEMANTIC VIEW`](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view), and [semantic SQL queries](https://docs.snowflake.com/en/sql-reference/constructs/semantic_view)

Feature behavior and syntax were checked against Snowflake documentation on 2026-09-07. Account edition, release rollout, region, role, and privileges can affect which sections execute.